# importing libraries

In [1]:
import os
os.chdir("..")

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader ,TensorDataset
import mlflow
import mlflow.pytorch
import optuna

import logging
logging.getLogger("mlflow").setLevel(logging.ERROR)

/Users/karimalmamlouk/Desktop/resources/github/Machine Failure Prediction with MLflow Tracking/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# load the datasets

In [2]:
# laod the training data as torch tensor
train = pd.read_csv(os.path.join("data" ,"train.csv"))
train_x = torch.tensor(train.drop(columns=["fail"]).values
                       ,dtype= torch.float32)
train_y = torch.tensor(train["fail"].values
                       ,dtype= torch.float32).unsqueeze(1)

# laod the testing data as torch tensor
test = pd.read_csv(os.path.join("data" ,"test.csv"))
test_x = torch.tensor(test.drop(columns=["fail"]).values
                      ,dtype= torch.float32)
test_y = torch.tensor(test["fail"].values
                      ,dtype= torch.float32).unsqueeze(1)

In [3]:
# craete datasets objects for training and testing
train_ds = TensorDataset(train_x ,train_y)
test_ds = TensorDataset(test_x ,test_y)

# craeting dataloader objects for training
batch_size = 64
train_loader = DataLoader(train_ds ,batch_size = batch_size ,shuffle = True)

# creating simple Neural Model Class

In [4]:
class Model(nn.Module):

    def __init__(self ,hidden_dim ,dropout_rate):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(train_x.shape[1] ,hidden_dim) ,
            nn.ReLU() ,
            nn.Dropout(p = dropout_rate),
            nn.Linear(hidden_dim ,1) ,
            nn.Sigmoid()
        )

    def forward(self ,x):
        return self.net(x)

# creating training pipeline

In [5]:
def train_pipeline(params):

    model = Model(params["hidden_dim"] 
                  ,params["dropout_rate"])
    loss_fn = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters() ,lr = params["lr"])

    # training loop
    for epoch in range(params["epochs"]):
        model.train()
        for x,y in train_loader:
            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred ,y)
            loss.backward()
            optimizer.step()
    
    # testing loop
    model.eval()
    with torch.no_grad():
        pred = model(test_x)
        test_loss = loss_fn(pred ,test_y)
        test_accu = ((pred > 0.5).float() == test_y).float().mean().item()
    
    return model ,test_loss ,test_accu

# using optuna and mlflow for hyperparameter tuning and training

In [6]:
# define objective function for optuna
def objective(trial):
    params = {
        "hidden_dim" : trial.suggest_int("hidden_dim" ,16 ,128) ,
        "dropout_rate" : trial.suggest_float("dropout_rate" ,0.1 ,0.3) ,
        "epochs" : trial.suggest_int("epochs" ,5 ,90) ,
        "lr" : trial.suggest_float("lr" ,0.01 ,0.15) 
    }

    # log using mlflow
    with mlflow.start_run():
        model ,test_loss ,test_accu = train_pipeline(params)
        mlflow.log_params(params)
        mlflow.log_metric("test_accuracy" ,test_accu)
        mlflow.log_metric("test_loss" ,test_loss)
        mlflow.pytorch.log_model(model, artifact_path="model")
        
    return test_loss

In [8]:
mlflow.set_experiment("test")
study = optuna.create_study(direction="minimize")
study.optimize(objective ,n_trials=50)

[I 2026-04-22 11:40:28,467] A new study created in memory with name: no-name-a0433f04-c25c-4221-8098-d4ab288cd460
[I 2026-04-22 11:40:28,860] Trial 0 finished with value: 0.7362388372421265 and parameters: {'hidden_dim': 112, 'dropout_rate': 0.2848806185103588, 'epochs': 43, 'lr': 0.05976380653493513}. Best is trial 0 with value: 0.7362388372421265.
[I 2026-04-22 11:40:29,370] Trial 1 finished with value: 0.8882428407669067 and parameters: {'hidden_dim': 94, 'dropout_rate': 0.2005022833575829, 'epochs': 70, 'lr': 0.02092422382925609}. Best is trial 0 with value: 0.7362388372421265.
[I 2026-04-22 11:40:29,560] Trial 2 finished with value: 0.4379782974720001 and parameters: {'hidden_dim': 21, 'dropout_rate': 0.2523354648332496, 'epochs': 11, 'lr': 0.14764483330680594}. Best is trial 2 with value: 0.4379782974720001.
[I 2026-04-22 11:40:29,943] Trial 3 finished with value: 0.8736280202865601 and parameters: {'hidden_dim': 72, 'dropout_rate': 0.1752656643413542, 'epochs': 47, 'lr': 0.08785